<a href="https://colab.research.google.com/github/AHMADAMJAD0/NLP_with_Olga/blob/main/Normalized__Text%2C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import re
import math
import numpy as np

from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [8]:
with open("text_news.txt", "r", encoding="utf-8", errors="ignore") as file:
    text = file.read()

print(text[:1000])


@@3000041 <p> Basketball United States Wins Bronze Medal : The United States used another comeback yesterday to take the bronze medal at the world championships in Buenos Aires , beating Puerto Rico by 107-105 in overtime . Kenny Anderson scored 34 points and made two free throws with one second left to send the game into overtime after Puerto Rico led by 8 points with 1 minute 34 seconds to play in the second half . Yugoslavia , which beat the United States in the semifinals , will face the Soviet Union for the gold medal today . This was the fourth straight international tournament in which the United States had failed to win a gold . Any medal for the United States looked out of the question with 1:34 left in regulation when Raymond Gausse 's 3-pointer gave Puerto Rico a 96-88 lead . Then Anderson hit a jumper with 1:21 to play to get the United States within 6 , and Puerto Rico opted for possession rather than foul shots four times in the final 1:19 . After a backcourt violation ,

In [9]:
documents = re.split(r'@@\d+', text)

# Remove empty documents
documents = [
    doc.strip()
    for doc in documents
    if doc.strip()
]

print("Number of documents:", len(documents))

Number of documents: 871


In [10]:
stop_words = set(ENGLISH_STOP_WORDS)

In [11]:
def normalize(text):

    # lowercase
    text = text.lower()

    # remove <p>
    text = text.replace("<p>", " ")

    # remove @ @ @ noise
    text = re.sub(r'(?:@\s*){2,}', ' ', text)

    # keep letters only
    text = re.sub(r'[^a-z\s]', ' ', text)

    # split into words
    words = text.split()

    # remove stopwords
    words = [
        word
        for word in words
        if word not in stop_words
    ]

    return words

In [12]:
example = "The students are studying in the university!"

print(normalize(example))

['students', 'studying', 'university']


In [13]:
normalized_documents = []

for doc in documents:

    words = normalize(doc)

    normalized_documents.append(words)

In [14]:
print(normalized_documents[0][:100])

['basketball', 'united', 'states', 'wins', 'bronze', 'medal', 'united', 'states', 'used', 'comeback', 'yesterday', 'bronze', 'medal', 'world', 'championships', 'buenos', 'aires', 'beating', 'puerto', 'rico', 'overtime', 'kenny', 'anderson', 'scored', 'points', 'free', 'throws', 'second', 'left', 'send', 'game', 'overtime', 'puerto', 'rico', 'led', 'points', 'minute', 'seconds', 'play', 'second', 'half', 'yugoslavia', 'beat', 'united', 'states', 'semifinals', 'face', 'soviet', 'union', 'gold', 'medal', 'today', 'fourth', 'straight', 'international', 'tournament', 'united', 'states', 'failed', 'win', 'gold', 'medal', 'united', 'states', 'looked', 'question', 'left', 'regulation', 'raymond', 'gausse', 's', 'pointer', 'gave', 'puerto', 'rico', 'lead', 'anderson', 'hit', 'jumper', 'play', 'united', 'states', 'puerto', 'rico', 'opted', 'possession', 'foul', 'shots', 'times', 'final', 'backcourt', 'violation', 'anderson', 'free', 'throws', 'seconds', 'left', 'todd', 'day', 'scored']


In [15]:
all_words = []

for doc in normalized_documents:

    all_words.extend(doc)

In [16]:
print("Total words:", len(all_words))

Total words: 602542


In [17]:
word_counts = Counter(all_words)

In [18]:
print(word_counts.most_common(20))

[('s', 13693), ('said', 5385), ('n', 4038), ('t', 3928), ('year', 2229), ('new', 2167), ('says', 2129), ('people', 1973), ('m', 1934), ('like', 1824), ('years', 1766), ('time', 1762), ('just', 1584), ('did', 1366), ('state', 1132), ('city', 1053), ('mr', 1009), ('day', 999), ('school', 988), ('p', 983)]


In [19]:
VOCAB_SIZE = 1000

vocabulary = [
    word
    for word, count
    in word_counts.most_common(VOCAB_SIZE)
]

In [20]:
print("Vocabulary size:", len(vocabulary))

print(vocabulary[:30])

Vocabulary size: 1000
['s', 'said', 'n', 't', 'year', 'new', 'says', 'people', 'm', 'like', 'years', 'time', 'just', 'did', 'state', 'city', 'mr', 'day', 'school', 'p', 'make', 'way', 'home', 'say', 'million', 'going', 'team', 'game', 'work', 'world']


In [21]:
word_to_id = {}

for i, word in enumerate(vocabulary):

    word_to_id[word] = i

In [22]:
print(word_to_id["time"])

11


In [23]:
WINDOW = 4

In [24]:
M = len(vocabulary)

context_matrix = np.zeros(
    (M, M),
    dtype=float
)

In [25]:
for doc in normalized_documents:

    for i in range(len(doc)):

        target_word = doc[i]

        # Ignore target words outside vocabulary
        if target_word not in word_to_id:
            continue

        target_id = word_to_id[target_word]

        # Left boundary
        start = max(0, i - WINDOW)

        # Right boundary
        end = min(len(doc), i + WINDOW + 1)

        for j in range(start, end):

            # Don't count the target word itself
            if j == i:
                continue

            context_word = doc[j]

            if context_word not in word_to_id:
                continue

            context_id = word_to_id[context_word]

            context_matrix[
                target_id,
                context_id
            ] += 1

In [26]:
time_id = word_to_id["time"]

time_vector = context_matrix[time_id]

In [27]:
top_context_ids = np.argsort(
    time_vector
)[::-1][:20]

In [28]:
for index in top_context_ids:

    print(
        vocabulary[index],
        time_vector[index]
    )

s 368.0
said 154.0
n 129.0
t 118.0
time 94.0
long 71.0
year 71.0
says 60.0
years 59.0
just 53.0
people 52.0
like 46.0
work 44.0
going 44.0
did 41.0
think 37.0
new 37.0
make 36.0
life 34.0
way 33.0


In [29]:
raw_tf = context_matrix.copy()

In [30]:
binary_tf = (
    context_matrix > 0
).astype(float)

In [31]:
log_tf = np.log1p(
    context_matrix
)

In [32]:
document_length = context_matrix.sum(
    axis=1
)

In [33]:
print(
    "Context length of time:",
    document_length[time_id]
)

Context length of time: 7150.0


In [34]:
average_document_length = (
    document_length.mean()
)

In [35]:
print(
    "Average context length:",
    average_document_length
)

Average context length: 1087.2


In [36]:
relative_frequency = np.zeros_like(
    context_matrix
)

In [37]:
for i in range(M):

    total = document_length[i]

    if total > 0:

        relative_frequency[i] = (
            context_matrix[i] / total
        )

In [38]:
document_frequency = (
    context_matrix > 0
).sum(axis=0)

In [39]:
number_of_documents = M

In [40]:
idf = np.log(
    (number_of_documents + 1)
    /
    (document_frequency + 1)
)

In [41]:
for word in ["time", "year", "government"]:

    if word in word_to_id:

        i = word_to_id[word]

        print(
            word,
            "IDF =",
            idf[i]
        )

time IDF = 0.05018974452385531
year IDF = 0.04704343883449041
government IDF = 0.3940420884426908


In [42]:
k = 1.5
b = 0.75

In [43]:
bm25 = np.zeros_like(
    context_matrix
)

In [44]:
for i in range(M):

    doc_len = document_length[i]

    for j in range(M):

        freq = context_matrix[i, j]

        if freq == 0:
            continue

        numerator = (
            (k + 1) * freq
        )

        denominator = (
            freq
            +
            k * (
                1 - b
                +
                b * (
                    doc_len
                    /
                    average_document_length
                )
            )
        )

        bm25[i, j] = (
            numerator / denominator
        )